<a href="https://colab.research.google.com/github/Sumandveer/CLAIMSCOPE-AI-INSURANCE-INVESTIGATION-MODEL/blob/main/CLAIMSCOPE_AI_INSURANCE_INVESTIGATION_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q spacy sentence-transformers faiss-cpu gradio scikit-learn torch pandas numpy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

vehicle_scenarios = [
    ("Collision", "I was driving home around 8:30 PM on {date_str} when another vehicle suddenly hit my car from behind near the main highway exit. Damage cost is estimated around {amt}.", "R14"),
    ("Collision", "My car was parked outside my residence on {date_str} and was struck on the rear bumper by an unknown vehicle. Repair shop quoted {amt}.", "R14"),
    ("Theft", "My vehicle was stolen from the commercial parking lot on {date_str} between 2 PM and 5 PM. Filed a police report immediately. Value claimed is {amt}.", "R02"),
    ("Weather", "Heavy hailstorm on {date_str} cracked my windshield and dented the hood while parked outdoors. Repair estimate is {amt}.", "R09"),
    ("Collision", "While changing lanes on {date_str}, a speeding truck clipped my front fender. No injuries reported, repair bill is {amt}.", "R05")
]

property_scenarios = [
    ("Water damage", "A main pipe burst in the kitchen wall on {date_str}, flooding the ground floor and ruining the wooden flooring. Loss is around {amt}.", "P08"),
    ("Burglary", "Returned home on {date_str} to find the back window broken and electronics missing from the living room. Total loss is {amt}.", "P03"),
    ("Fire", "An electrical short circuit in the utility room caused a fire on {date_str}, damaging appliances and drywall. Repair cost is {amt}.", "P11"),
    ("Water damage", "Heavy rain leaked through the roof on {date_str} and damaged ceiling plaster and bedroom furniture. Estimated damage is {amt}.", "P08"),
    ("Vandalism", "Unknown individuals broke the front glass doors of my shop on {date_str} overnight. Replacement cost is {amt}.", "P04")
]

records = []
base_date = datetime(2026, 1, 1)

for i in range(1, 601):
    claim_id = f"CLM_{i:04d}"
    category = "Vehicle" if i <= 300 else "Property"
    scenarios = vehicle_scenarios if category == "Vehicle" else property_scenarios

    inc_type, template, default_provider = random.choice(scenarios)

    inc_date = base_date + timedelta(days=random.randint(0, 280))
    inc_date_str = inc_date.strftime("%Y-%m-%d")

    form_amount = random.randint(15000, 150000)
    policy_limit = form_amount + random.randint(20000, 100000)

    pattern_type = random.choices(
        ["Genuine", "Accidental_Inconsistency", "Date_Mismatch", "Inflated_Amount", "Organized_Ring"],
        weights=[0.65, 0.10, 0.08, 0.09, 0.08]
    )[0]

    narrative_date_str = inc_date_str
    narrative_amount = form_amount
    provider = default_provider if pattern_type == "Organized_Ring" else f"P{random.randint(1, 25):02d}"
    report_delay = random.randint(1, 4)
    is_fraud = 0
    evidence = "All details match supporting records."

    if pattern_type == "Accidental_Inconsistency":
        narrative_date_str = (inc_date + timedelta(days=1)).strftime("%Y-%m-%d")
        is_fraud = 0
        evidence = "Minor 1-day date typo in narrative; otherwise consistent."

    elif pattern_type == "Date_Mismatch":
        narrative_date_str = (inc_date + timedelta(days=random.randint(5, 15))).strftime("%Y-%m-%d")
        is_fraud = 1
        evidence = f"Form date ({inc_date_str}) conflicts sharply with narrative date ({narrative_date_str})."

    elif pattern_type == "Inflated_Amount":
        narrative_amount = int(form_amount * 0.55)
        is_fraud = 1
        evidence = f"Form claimed amount ({form_amount}) is nearly double the narrative estimate ({narrative_amount})."

    elif pattern_type == "Organized_Ring":
        narrative = f"My vehicle was struck from behind while parked outside my residence on {inc_date_str}. Repair estimate is {form_amount}." if category == "Vehicle" else f"A main pipe burst in the kitchen wall on {inc_date_str}, flooding the ground floor. Loss is {form_amount}."
        provider = "R14" if category == "Vehicle" else "P08"
        is_fraud = 1
        evidence = f"Repeated near-identical narrative linked to provider {provider}."
    else:
        narrative = template.format(date_str=narrative_date_str, amt=narrative_amount)

    if pattern_type != "Organized_Ring":
        narrative = template.format(date_str=narrative_date_str, amt=narrative_amount)

    report_date = (inc_date + timedelta(days=report_delay)).strftime("%Y-%m-%d")

    records.append({
        "claim_id": claim_id,
        "category": category,
        "incident_type": inc_type,
        "form_incident_date": inc_date_str,
        "report_date": report_date,
        "form_claimed_amount": form_amount,
        "policy_limit": policy_limit,
        "provider_id": provider,
        "narrative": narrative,
        "fraud_pattern": pattern_type,
        "label": is_fraud,
        "evidence_notes": evidence
    })

df = pd.DataFrame(records)
df.to_csv("claimscope_master.csv", index=False)
print(f"Dataset created: {len(df)} rows | Fraud cases: {df['label'].sum()}")
df.head(5)

Dataset created: 600 rows | Fraud cases: 139


,claim_id,category,incident_type,form_incident_date,report_date,form_claimed_amount,policy_limit,provider_id,narrative,fraud_pattern,label,evidence_notes
0,CLM_0001,Vehicle,Collision,2026-01-13,2026-01-14,87097,139195,P24,I was driving home around 8:30 PM on 2026-01-1...,Genuine,0,All details match supporting records.
1,CLM_0002,Vehicle,Collision,2026-02-14,2026-02-16,125604,149769,P07,"While changing lanes on 2026-02-14, a speeding...",Genuine,0,All details match supporting records.
2,CLM_0003,Vehicle,Collision,2026-01-14,2026-01-17,67124,158550,P15,"While changing lanes on 2026-01-14, a speeding...",Genuine,0,All details match supporting records.
3,CLM_0004,Vehicle,Collision,2026-03-23,2026-03-26,125785,190382,P07,I was driving home around 8:30 PM on 2026-03-2...,Genuine,0,All details match supporting records.
4,CLM_0005,Vehicle,Collision,2026-02-17,2026-02-20,114595,147271,P12,I was driving home around 8:30 PM on 2026-02-1...,Genuine,0,All details match supporting records.


In [3]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def extract_and_check_consistency(narrative, form_date, report_date, form_amount, policy_limit):
    doc = nlp(narrative)

    extracted_dates = [ent.text for ent in doc.ents if ent.label_ == "DATE"]

    date_matches = re.findall(r"\d{4}-\d{2}-\d{2}", narrative)
    extracted_date = date_matches[0] if date_matches else (extracted_dates[0] if extracted_dates else "Not found")

    numbers = [int(n) for n in re.findall(r"\b\d{4,7}\b", narrative)]
    extracted_amount = numbers[0] if numbers else form_amount

    flags = []
    consistency_score = 0

    if extracted_date != "Not found" and extracted_date != form_date:
        flags.append(f"Date Conflict: Form states {form_date}, narrative mentions {extracted_date}.")
        consistency_score += 40

    if report_date < form_date:
        flags.append(f"Timeline Error: Claim reported ({report_date}) before incident date ({form_date}).")
        consistency_score += 50

    if abs(form_amount - extracted_amount) > (0.20 * form_amount):
        flags.append(f"Amount Discrepancy: Form claims {form_amount}, narrative states {extracted_amount}.")
        consistency_score += 35

    if form_amount > policy_limit:
        flags.append(f"Policy Limit Exceeded: Claimed {form_amount} > Limit {policy_limit}.")
        consistency_score += 30

    return {
        "extracted_date": extracted_date,
        "extracted_amount": extracted_amount,
        "consistency_risk": min(consistency_score, 100),
        "flags": flags
    }

sample = df.iloc[0]
print(extract_and_check_consistency(
    sample["narrative"], sample["form_incident_date"], sample["report_date"],
    sample["form_claimed_amount"], sample["policy_limit"]
))

{'extracted_date': '2026-01-13', 'extracted_amount': 2026, 'consistency_risk': 35, 'flags': ['Amount Discrepancy: Form claims 87097, narrative states 2026.']}


In [4]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
narrative_embeddings = embedder.encode(df["narrative"].tolist(), convert_to_numpy=True)
faiss.normalize_L2(narrative_embeddings)

dimension = narrative_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(narrative_embeddings)

def find_similar_claims(query_narrative, provider_id, top_k=3):
    q_emb = embedder.encode([query_narrative], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    scores, indices = index.search(q_emb, top_k)
    matches = []
    max_sim = 0.0

    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        sim_pct = round(float(score) * 100, 1)
        if sim_pct > max_sim:
            max_sim = sim_pct
        matches.append({
            "claim_id": row["claim_id"],
            "similarity": f"{sim_pct}%",
            "provider": row["provider_id"],
            "narrative": row["narrative"]
        })

    provider_count = len(df[df["provider_id"] == provider_id])
    return matches, max_sim, provider_count

print("FAISS Index built with", index.ntotal, "claims.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index built with 600 claims.


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df["narrative"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

# 1. Traditional NLP Baseline: TF-IDF + Logistic Regression
tfidf = TfidfVectorizer(max_features=500, stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_model = LogisticRegression(class_weight="balanced")
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print("--- Model 1: TF-IDF + Logistic Regression ---")
print(classification_report(y_test, lr_preds, zero_division=0))

# 2. Deep Learning Model: PyTorch LSTM
vocab = {"<PAD>": 0, "<UNK>": 1}
for text in X_train:
    for word in text.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

def encode_text(texts, max_len=35):
    encoded = []
    for text in texts:
        tokens = [vocab.get(w, 1) for w in text.lower().split()[:max_len]]
        tokens += [0] * (max_len - len(tokens))
        encoded.append(tokens)
    return torch.tensor(encoded, dtype=torch.long)

X_train_seq = encode_text(X_train)
X_test_seq = encode_text(X_test)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_seq, y_train_t), batch_size=32, shuffle=True)

class ClaimLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        _, (hidden, _) = self.lstm(emb)
        return torch.sigmoid(self.fc(hidden[-1])).squeeze(1)

lstm_model = ClaimLSTM(len(vocab))
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)
criterion = nn.BCELoss()

for epoch in range(8):
    lstm_model.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = lstm_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()

lstm_model.eval()
with torch.no_grad():
    lstm_probs = lstm_model(X_test_seq)
    lstm_preds = (lstm_probs > 0.5).int().numpy()

print("--- Model 2: PyTorch LSTM ---")
print(classification_report(y_test, lstm_preds, zero_division=0))

--- Model 1: TF-IDF + Logistic Regression ---
              precision    recall  f1-score   support

           0       0.81      0.92      0.86        92
           1       0.53      0.29      0.37        28

    accuracy                           0.78       120
   macro avg       0.67      0.60      0.62       120
weighted avg       0.75      0.78      0.75       120

--- Model 2: PyTorch LSTM ---
              precision    recall  f1-score   support

           0       0.79      1.00      0.88        92
           1       1.00      0.11      0.19        28

    accuracy                           0.79       120
   macro avg       0.89      0.55      0.54       120
weighted avg       0.84      0.79      0.72       120



In [6]:
import gradio as gr

def investigate_claim(category, form_date, report_date, form_amount, policy_limit, provider_id, narrative):
    # 1. Consistency Check
    consistency = extract_and_check_consistency(
        narrative, form_date, report_date, float(form_amount), float(policy_limit)
    )

    # 2. Historical Similarity Search
    similar_claims, max_sim, provider_freq = find_similar_claims(narrative, provider_id)

    # 3. NLP Model Score
    vec = tfidf.transform([narrative])
    nlp_prob = round(float(lr_model.predict_proba(vec)[0][1]) * 100, 1)

    # 4. Hybrid Investigation Priority Score
    sim_risk = 30 if max_sim > 92 else (15 if max_sim > 85 else 0)
    provider_risk = 20 if provider_freq > 35 else 0

    total_score = min(int(0.45 * consistency["consistency_risk"] + 0.30 * nlp_prob + sim_risk + provider_risk), 100)

    if total_score >= 65:
        priority = "HIGH REVIEW PRIORITY"
        action = "Escalate to Senior Investigator. Verify supporting documents and cross-check linked historical claims."
    elif total_score >= 35:
        priority = "ELEVATED REVIEW PRIORITY"
        action = "Request claimant clarification on flagged discrepancies before approving payout."
    else:
        priority = "STANDARD PROCESSING"
        action = "No major inconsistencies or suspicious patterns detected. Proceed with standard verification."

    # Format Evidence Report
    report = f"## Investigation Priority: {priority} ({total_score}/100)\n"
    report += f"**Recommended Action:** {action}\n\n"

    report += "### 1. Extracted Entities & Consistency Checks\n"
    report += f"- **Extracted Narrative Date:** `{consistency['extracted_date']}` (Form: `{form_date}`)\n"
    report += f"- **Extracted Narrative Amount:** `{consistency['extracted_amount']}` (Form: `{form_amount}`)\n"
    if consistency["flags"]:
        for flag in consistency["flags"]:
            report += f"- ⚠️ **{flag}**\n"
    else:
        report += "- ✅ No cross-document date or amount conflicts found.\n"

    report += "\n### 2. Historical Pattern & Entity Intelligence\n"
    report += f"- **Repairer/Provider (`{provider_id}`) History:** Appears in **{provider_freq}** historical claims.\n"
    report += f"- **NLP Pattern Risk Indicator:** `{nlp_prob}%`\n\n"

    report += "### 3. Top Similar Historical Claims (FAISS)\n"
    for m in similar_claims:
        report += f"- **{m['claim_id']}** ({m['similarity']} match | Provider: `{m['provider']}`): _{m['narrative']}_\n"

    return report

demo = gr.Interface(
    fn=investigate_claim,
    inputs=[
        gr.Dropdown(["Vehicle", "Property"], value="Vehicle", label="Insurance Category"),
        gr.Textbox(value="2026-08-10", label="Form Incident Date (YYYY-MM-DD)"),
        gr.Textbox(value="2026-08-12", label="Claim Report Date (YYYY-MM-DD)"),
        gr.Number(value=95000, label="Form Claimed Amount"),
        gr.Number(value=150000, label="Policy Coverage Limit"),
        gr.Textbox(value="R14", label="Repairer / Provider ID"),
        gr.Textbox(
            lines=3,
            value="My vehicle was struck from behind while parked outside my residence on 2026-08-22. Repair estimate is 45000.",
            label="Claimant Written Explanation"
        )
    ],
    outputs=gr.Markdown(label="CLAIMSCOPE Investigation Report"),
    title="CLAIMSCOPE — AI Insurance Claim Investigation Platform",
    description="Analyzes claimant narratives, checks cross-source consistency, and discovers recurring historical patterns."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://089a9c7deb728ae9b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
